# Titanic Threshold Animation: ROC Curve & PR Curve

이 노트북은 Titanic 데이터를 기반으로 아래 4개 모델을 학습하고, threshold 변화에 따라 ROC Curve / Precision-Recall Curve 위의 위치가 어떻게 이동하는지 애니메이션으로 시각화합니다.

- Logistic Regression
- Decision Tree
- Random Forest
- MLP Classifier

## 핵심 학습 포인트

- threshold는 모델 자체가 아니라 **의사결정 정책 레버**입니다.
- threshold를 낮추면 positive class 예측이 많아져 recall이 높아지는 경향이 있습니다.
- threshold를 높이면 더 확실한 케이스만 positive로 예측해 precision이 높아지는 경향이 있습니다.
- ROC Curve와 PR Curve에서 threshold별 위치를 같이 보면 classification policy trade-off를 직관적으로 이해할 수 있습니다.

## 1. Library Import

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    auc,
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

RANDOM_STATE = 42

## 2. Titanic 데이터 로드

아래 셀은 우선 로컬에 `train.csv`가 있으면 Kaggle Titanic 데이터를 사용합니다.  
없으면 seaborn의 Titanic 데이터를 자동으로 불러옵니다.

In [2]:
try:
    df = pd.read_csv("train.csv")
    print("Loaded Kaggle Titanic train.csv")

    required_cols = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
    df = df[required_cols].copy()

except FileNotFoundError:
    print("train.csv not found. Loading seaborn Titanic dataset instead.")
    import seaborn as sns

    df = sns.load_dataset("titanic")
    df = df.rename(columns={
        "survived": "Survived",
        "pclass": "Pclass",
        "sex": "Sex",
        "age": "Age",
        "sibsp": "SibSp",
        "parch": "Parch",
        "fare": "Fare",
        "embarked": "Embarked"
    })

    df = df[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()

df.head()

train.csv not found. Loading seaborn Titanic dataset instead.


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


## 3. Feature / Target 정의

In [3]:
target_col = "Survived"

feature_cols = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

X = df[feature_cols]
y = df[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Positive ratio in test:", y_test.mean().round(3))

Train shape: (668, 7)
Test shape: (223, 7)
Positive ratio in test: 0.386


## 4. 전처리 파이프라인

- Numeric columns: median imputation + standard scaling
- Categorical columns: most frequent imputation + one-hot encoding

In [4]:
numeric_features = ["Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Pclass", "Sex", "Embarked"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 5. 모델 정의 및 학습

In [5]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=4,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        random_state=RANDOM_STATE
    ),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        max_iter=1000,
        random_state=RANDOM_STATE
    )
}

trained_models = {}
y_score_dict = {}

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    trained_models[model_name] = pipeline

    # Positive class probability: P(Survived = 1)
    y_score = pipeline.predict_proba(X_test)[:, 1]
    y_score_dict[model_name] = y_score

print("Training completed.")

Training completed.


## 6. ROC / PR Curve 데이터 계산

In [6]:
metrics_dict = {}

for model_name, y_score in y_score_dict.items():
    fpr, tpr, roc_thresholds = roc_curve(y_test, y_score)
    precision, recall, pr_thresholds = precision_recall_curve(y_test, y_score)

    roc_auc = auc(fpr, tpr)
    avg_precision = average_precision_score(y_test, y_score)

    metrics_dict[model_name] = {
        "fpr": fpr,
        "tpr": tpr,
        "roc_thresholds": roc_thresholds,
        "precision": precision,
        "recall": recall,
        "pr_thresholds": pr_thresholds,
        "roc_auc": roc_auc,
        "avg_precision": avg_precision,
    }

summary_rows = []
for model_name, data in metrics_dict.items():
    summary_rows.append({
        "model": model_name,
        "roc_auc": data["roc_auc"],
        "average_precision": data["avg_precision"]
    })

model_summary = pd.DataFrame(summary_rows).sort_values("roc_auc", ascending=False)
model_summary

,model,roc_auc,average_precision
0,Logistic Regression,0.842090,0.784733
2,Random Forest,0.840562,0.821786
1,Decision Tree,0.810177,0.741962
3,MLP,0.801689,0.773807


## 7. Threshold별 성능 테이블 생성

threshold를 `0.00 ~ 1.00`까지 변화시키면서 Precision, Recall, F1, FPR, TPR, Confusion Matrix 값을 계산합니다.

In [7]:
thresholds = np.linspace(0.0, 1.0, 101)

threshold_result_rows = []

for model_name, y_score in y_score_dict.items():
    for threshold in thresholds:
        y_pred = (y_score >= threshold).astype(int)

        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        fpr_value = fp / (fp + tn) if (fp + tn) > 0 else 0
        tpr_value = tp / (tp + fn) if (tp + fn) > 0 else 0

        threshold_result_rows.append({
            "model": model_name,
            "threshold": threshold,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "fpr": fpr_value,
            "tpr": tpr_value,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp
        })

threshold_results = pd.DataFrame(threshold_result_rows)
threshold_results.head()

,model,threshold,precision,recall,f1,fpr,tpr,tn,fp,fn,tp
0,Logistic Regression,0.00,0.385650,1.0,0.556634,1.000000,1.0,0,137,0,86
1,Logistic Regression,0.01,0.385650,1.0,0.556634,1.000000,1.0,0,137,0,86
2,Logistic Regression,0.02,0.387387,1.0,0.558442,0.992701,1.0,1,136,0,86
3,Logistic Regression,0.03,0.387387,1.0,0.558442,0.992701,1.0,1,136,0,86
4,Logistic Regression,0.04,0.390909,1.0,0.562092,0.978102,1.0,3,134,0,86


## 8. 모델별 Best F1 Threshold 확인

In [8]:
best_f1_rows = threshold_results.loc[
    threshold_results.groupby("model")["f1"].idxmax()
].sort_values("f1", ascending=False)

best_f1_rows[
    ["model", "threshold", "precision", "recall", "f1", "fpr", "tpr", "tn", "fp", "fn", "tp"]
]

,model,threshold,precision,recall,f1,fpr,tpr,tn,fp,fn,tp
40,Logistic Regression,0.40,0.711340,0.802326,0.754098,0.204380,0.802326,109,28,17,69
237,Random Forest,0.35,0.660550,0.837209,0.738462,0.270073,0.837209,100,37,14,72
333,MLP,0.30,0.683673,0.779070,0.728261,0.226277,0.779070,106,31,19,67
112,Decision Tree,0.11,0.590164,0.837209,0.692308,0.364964,0.837209,87,50,14,72


## 9. 단일 모델 Threshold Animation

아래 `selected_model` 값을 바꿔가며 실행할 수 있습니다.

사용 가능 모델명:

- `"Logistic Regression"`
- `"Decision Tree"`
- `"Random Forest"`
- `"MLP"`

In [9]:
selected_model = "Random Forest"

model_result = threshold_results[
    threshold_results["model"] == selected_model
].reset_index(drop=True)

roc_data = metrics_dict[selected_model]

print(f"Selected model: {selected_model}")

Selected model: Random Forest


In [10]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax_roc = axes[0, 0]
ax_pr = axes[0, 1]
ax_metric = axes[1, 0]
ax_cm = axes[1, 1]

# -----------------------------
# Static ROC Curve
# -----------------------------
ax_roc.plot(
    roc_data["fpr"],
    roc_data["tpr"],
    label=f"ROC AUC = {roc_data['roc_auc']:.3f}"
)
ax_roc.plot([0, 1], [0, 1], linestyle="--")
roc_point, = ax_roc.plot([], [], marker="o", markersize=10)

ax_roc.set_title(f"{selected_model} - ROC Curve")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.legend(loc="lower right")
ax_roc.grid(True)

# -----------------------------
# Static PR Curve
# -----------------------------
ax_pr.plot(
    roc_data["recall"],
    roc_data["precision"],
    label=f"AP = {roc_data['avg_precision']:.3f}"
)
pr_point, = ax_pr.plot([], [], marker="o", markersize=10)

ax_pr.set_title(f"{selected_model} - Precision-Recall Curve")
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.legend(loc="lower left")
ax_pr.grid(True)

# -----------------------------
# Static Metric Lines
# -----------------------------
ax_metric.plot(model_result["threshold"], model_result["precision"], label="Precision")
ax_metric.plot(model_result["threshold"], model_result["recall"], label="Recall")
ax_metric.plot(model_result["threshold"], model_result["f1"], label="F1")
threshold_line = ax_metric.axvline(0, linestyle="--")

ax_metric.set_title("Metrics by Threshold")
ax_metric.set_xlabel("Threshold")
ax_metric.set_ylabel("Score")
ax_metric.set_ylim(0, 1.05)
ax_metric.legend()
ax_metric.grid(True)

# -----------------------------
# Confusion Matrix Initial
# -----------------------------
cm_image = ax_cm.imshow(np.zeros((2, 2)))
cm_texts = []

for i in range(2):
    row_texts = []
    for j in range(2):
        text = ax_cm.text(
            j,
            i,
            "",
            ha="center",
            va="center",
            fontsize=16
        )
        row_texts.append(text)
    cm_texts.append(row_texts)

ax_cm.set_title("Confusion Matrix")
ax_cm.set_xticks([0, 1])
ax_cm.set_yticks([0, 1])
ax_cm.set_xticklabels(["Pred 0", "Pred 1"])
ax_cm.set_yticklabels(["Actual 0", "Actual 1"])

fig.suptitle("", fontsize=16)


def update_single_model(frame):
    row = model_result.iloc[frame]

    threshold = row["threshold"]
    precision = row["precision"]
    recall = row["recall"]
    f1 = row["f1"]
    fpr_value = row["fpr"]
    tpr_value = row["tpr"]

    tn = int(row["tn"])
    fp = int(row["fp"])
    fn = int(row["fn"])
    tp = int(row["tp"])

    # ROC point
    roc_point.set_data([fpr_value], [tpr_value])

    # PR point
    pr_point.set_data([recall], [precision])

    # Threshold vertical line
    threshold_line.set_xdata([threshold, threshold])

    # Confusion matrix
    cm = np.array([
        [tn, fp],
        [fn, tp]
    ])

    cm_image.set_data(cm)
    cm_image.set_clim(vmin=0, vmax=cm.max() if cm.max() > 0 else 1)

    labels = [
        ["TN", "FP"],
        ["FN", "TP"]
    ]

    for i in range(2):
        for j in range(2):
            cm_texts[i][j].set_text(f"{labels[i][j]}\n{cm[i, j]}")

    fig.suptitle(
        f"{selected_model} | Threshold = {threshold:.2f} | "
        f"Precision = {precision:.3f}, Recall = {recall:.3f}, F1 = {f1:.3f}",
        fontsize=15
    )

    return [
        roc_point,
        pr_point,
        threshold_line,
        cm_image,
        *[text for row_text in cm_texts for text in row_text]
    ]


ani = FuncAnimation(
    fig,
    update_single_model,
    frames=len(model_result),
    interval=120,
    blit=False,
    repeat=True
)

plt.close(fig)

HTML(ani.to_jshtml())

Output hidden; open in https://colab.research.google.com to view.

## 10. 여러 모델 동시 비교 Animation

4개 모델의 threshold 위치를 ROC Curve / PR Curve에서 동시에 확인합니다.

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax_roc = axes[0]
ax_pr = axes[1]

roc_points = {}
pr_points = {}

for model_name in models.keys():
    data = metrics_dict[model_name]

    ax_roc.plot(
        data["fpr"],
        data["tpr"],
        label=f"{model_name} AUC={data['roc_auc']:.3f}"
    )
    point, = ax_roc.plot([], [], marker="o", markersize=8)
    roc_points[model_name] = point

    ax_pr.plot(
        data["recall"],
        data["precision"],
        label=f"{model_name} AP={data['avg_precision']:.3f}"
    )
    point, = ax_pr.plot([], [], marker="o", markersize=8)
    pr_points[model_name] = point

ax_roc.plot([0, 1], [0, 1], linestyle="--")
ax_roc.set_title("ROC Curve - Threshold Movement")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.legend()
ax_roc.grid(True)

ax_pr.set_title("Precision-Recall Curve - Threshold Movement")
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.legend()
ax_pr.grid(True)

fig.suptitle("", fontsize=16)


def update_multi_model(frame):
    threshold = thresholds[frame]

    for model_name in models.keys():
        row = threshold_results[
            (threshold_results["model"] == model_name) &
            (np.isclose(threshold_results["threshold"], threshold))
        ].iloc[0]

        roc_points[model_name].set_data(
            [row["fpr"]],
            [row["tpr"]]
        )

        pr_points[model_name].set_data(
            [row["recall"]],
            [row["precision"]]
        )

    fig.suptitle(
        f"Threshold = {threshold:.2f}",
        fontsize=16
    )

    return list(roc_points.values()) + list(pr_points.values())


ani_multi = FuncAnimation(
    fig,
    update_multi_model,
    frames=len(thresholds),
    interval=120,
    blit=False,
    repeat=True
)

plt.close(fig)

HTML(ani_multi.to_jshtml())

Output hidden; open in https://colab.research.google.com to view.

## 11. GIF / MP4 저장

환경에 따라 writer 설치가 필요할 수 있습니다.

- GIF 저장: `pillow`
- MP4 저장: `ffmpeg`

In [ ]:
# GIF 저장
# ani.save("titanic_threshold_single_model_animation.gif", writer="pillow", fps=8)

# 여러 모델 비교 GIF 저장
# ani_multi.save("titanic_threshold_multi_model_animation.gif", writer="pillow", fps=8)

# MP4 저장
# ani.save("titanic_threshold_single_model_animation.mp4", writer="ffmpeg", fps=8)
# ani_multi.save("titanic_threshold_multi_model_animation.mp4", writer="ffmpeg", fps=8)

## 12. 해석 가이드

| 관점 | Threshold 낮음 | Threshold 높음 |
|---|---:|---:|
| Positive 예측 수 | 많음 | 적음 |
| Recall | 높아지는 경향 | 낮아지는 경향 |
| Precision | 낮아질 수 있음 | 높아질 수 있음 |
| ROC 위치 | FPR과 TPR이 함께 높아지는 방향 | FPR과 TPR이 함께 낮아지는 방향 |
| PR 위치 | Recall 높은 구간 | Precision 높은 구간 |
| 정책 해석 | 놓치지 않는 전략 | 확실한 케이스만 잡는 전략 |

Titanic 데이터에서는 `Survived=1`을 positive class로 두었습니다.  
따라서 threshold를 낮추면 더 많은 승객을 생존으로 예측하고, threshold를 높이면 모델이 더 확신하는 경우만 생존으로 예측합니다.